# 09 — ATT&CK Knowledge Base & Malicious Event Index

Two artefacts produced here feed the SLM attribution pipeline (notebook 10):

1. **`checkpoints/events_m.parquet`** — one row per event in `X_m`, same order as
   the feature matrices.  Contains the human-readable Sysmon fields the SLM needs
   (`image`, `command_line`, `parent_image`, etc.) plus `attck_technique` where known.

2. **ChromaDB vector store** — ATT&CK technique descriptions indexed by
   `sentence-transformers/all-MiniLM-L6-v2`, used for RAG retrieval.

The malicious event loading order **must match notebook 05 exactly**:
`LMD malicious → OTRF Atomic → OTRF Compound → Splunk`

## Imports

In [1]:
import gc
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Make project modules importable from the notebooks/ directory
sys.path.insert(0, str(Path('..').resolve()))

from config import ATTACK_KB_DIR, CHROMA_PERSIST_DIR, EMBEDDING_MODEL, RAG_TOP_K
from data.ingest.lmd2023 import iter_chunks as lmd_iter
from data.ingest.otrf    import iter_compound, iter_atomic
from data.ingest.splunk  import iter_techniques as splunk_iter

warnings.filterwarnings('ignore')

CKPT_DIR        = Path('checkpoints')
EVENTS_M_PATH   = CKPT_DIR / 'events_m.parquet'

# Sysmon fields kept for SLM context — covers all major event types
EVENT_FIELDS = [
    'event_id', 'utc_time',
    'image', 'command_line',
    'parent_image', 'parent_cmdline',
    # EID=7: image_loaded = which DLL was loaded (key for T1218, T1574 etc.)
    'image_loaded',
    # EID=13: details = value written to registry (key for T1562 etc.)
    'details',
    # EID=10: granted_access = access rights requested (key for T1003 etc.)
    'granted_access',
    'target_object', 'target_image',
    'dest_ip', 'dest_hostname',
    'target_filename', 'query_name',
    'hashes',
    'attck_technique', 'dataset_source',
]

print('Ready.')
print(f'events_m target : {EVENTS_M_PATH}')
print(f'ChromaDB path   : {CHROMA_PERSIST_DIR}')

Ready.
events_m target : checkpoints\events_m.parquet
ChromaDB path   : H:\Challenge-3-4\cyber-anomaly-detection\data\attack_kb\chroma_db


## 1. Build malicious event index (`events_m.parquet`)

In [2]:
# Re-parse malicious events in the exact same order as notebook 05.
# We only keep the SLM-relevant fields — no feature encoding.

def _keep(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in EVENT_FIELDS if c in df.columns]
    return df[cols].copy()

parts = []

# ── LMD malicious (label > 0) ────────────────────────────────────────────────
print('Loading LMD malicious...')
for chunk in lmd_iter(variant='2.3M'):
    mal = chunk[chunk['label'] > 0]
    if not mal.empty:
        parts.append(_keep(mal))
n_lmd = sum(len(p) for p in parts)
print(f'  {n_lmd:,} events')

# ── OTRF Atomic ───────────────────────────────────────────────────────────────
print('Loading OTRF Atomic...')
n_before = sum(len(p) for p in parts)
for chunk in iter_atomic():
    mal = chunk[chunk['label'] == 1]
    if not mal.empty:
        parts.append(_keep(mal))
n_atomic = sum(len(p) for p in parts) - n_before
print(f'  {n_atomic:,} events')

# ── OTRF Compound ─────────────────────────────────────────────────────────────
print('Loading OTRF Compound...')
n_before = sum(len(p) for p in parts)
for chunk in iter_compound():
    parts.append(_keep(chunk))
n_compound = sum(len(p) for p in parts) - n_before
print(f'  {n_compound:,} events')

# ── Splunk ────────────────────────────────────────────────────────────────────
print('Loading Splunk techniques...')
n_before = sum(len(p) for p in parts)
for chunk in splunk_iter():
    parts.append(_keep(chunk))
n_splunk = sum(len(p) for p in parts) - n_before
print(f'  {n_splunk:,} events')

events_m = pd.concat(parts, ignore_index=True)
print(f'\nTotal events_m  : {len(events_m):,}')
del parts
gc.collect()

Loading LMD malicious...
  511,105 events
Loading OTRF Atomic...
  543,077 events
Loading OTRF Compound...
  328,019 events
Loading Splunk techniques...
  2,065,466 events

Total events_m  : 3,447,667


0

In [3]:
# Verify row count matches X_m feature matrix
X_m_check = np.load(CKPT_DIR / 'word2vec' / 'X_m_w2v.npy', mmap_mode='r')
assert len(events_m) == X_m_check.shape[0], (
    f'Row count mismatch: events_m={len(events_m):,}  X_m={X_m_check.shape[0]:,}\n'
    f'Ensure malicious loading order matches notebook 05 exactly.'
)
del X_m_check
print(f'Row count matches X_m: {len(events_m):,} OK')

# Save
EVENTS_M_PATH.parent.mkdir(parents=True, exist_ok=True)
events_m.to_parquet(EVENTS_M_PATH, index=False, engine='pyarrow', compression='snappy')
size_mb = EVENTS_M_PATH.stat().st_size / 1e6
print(f'Saved: {EVENTS_M_PATH}  ({size_mb:.1f} MB)')

# Per-source breakdown
print('\nPer-source breakdown:')
for src, grp in events_m.groupby('dataset_source', sort=False):
    n_with_tcode = (grp['attck_technique'] != '').sum()
    tcodes = grp['attck_technique'].replace('', pd.NA).dropna().nunique()
    print(f'  {src:<18} {len(grp):>9,} events   T-codes: {tcodes:>4}  '
          f'({n_with_tcode/len(grp)*100:.1f}% labelled)')

Row count matches X_m: 3,447,667 OK
Saved: checkpoints\events_m.parquet  (82.8 MB)

Per-source breakdown:
  LMD                  511,105 events   T-codes:    0  (0.0% labelled)
  OTRF-Atomic          543,077 events   T-codes:   53  (100.0% labelled)
  OTRF-Log4Shell           487 events   T-codes:    1  (100.0% labelled)
  OTRF-LSASS           327,532 events   T-codes:    1  (100.0% labelled)
  Splunk             2,065,466 events   T-codes:  177  (100.0% labelled)


In [4]:
# Sanity check — show one event per source
print('Sample events per source:\n')
for src, grp in events_m.groupby('dataset_source', sort=False):
    row = grp.iloc[0]
    print(f'[{src}]  T={row.get("attck_technique","?")}')
    for f in ['event_id','image','command_line','parent_image','target_object']:
        val = str(row.get(f,''))
        if val and val not in ('', '-1', 'nan'):
            print(f'  {f:<20}: {val[:100]}')
    print()

Sample events per source:

[LMD]  T=
  event_id            : 1
  image               : C:\Windows\System32\svchost.exe
  command_line        : C:\Windows\System32\svchost.exe -k LocalServiceNetworkRestricted -p
  parent_image        : -
  target_object       : 0

[OTRF-Atomic]  T=T1123
  event_id            : 13
  image               : C:\Windows\SystemApps\Microsoft.Windows.Cortana_cw5n1h2txyewy\SearchUI.exe
  target_object       : \REGISTRY\A\{361c92d4-b0d9-fc34-0cf8-30ca2aeb8b9b}\LocalState\AppsConstraintIndex\LastConstraintInde

[OTRF-Log4Shell]  T=T1190
  event_id            : 3

[OTRF-LSASS]  T=T1003.001
  event_id            : 10
  image               : C:\Windows\System32\VBoxService.exe

[Splunk]  T=T1003
  event_id            : 1
  image               : C:\Users\Administrator\Downloads\mimikatz_trunk\x64\mimikatz.exe
  command_line        : mimikatz.exe
  parent_image        : C:\Windows\System32\cmd.exe



## 2. Build ATT&CK knowledge base

In [5]:
import sys, os
# data/attack_kb/ is one level up from notebooks/
sys.path.insert(0, str(Path('..').resolve()))

from data.attack_kb.builder      import build_kb
from data.attack_kb.vector_store import ingest, get_collection

print('Building ATT&CK KB entries...')
entries = build_kb(force=True)    # force=True to rebuild with Sigma rules included
print(f'  {len(entries)} entries loaded')

# Technique coverage summary
from collections import Counter
source_counts = Counter(e.source for e in entries)
print('\nEntries per source:')
for src, n in sorted(source_counts.items()):
    print(f'  {src:<20} {n:>4}')

tcodes = {e.technique_id for e in entries}
print(f'\nUnique T-codes: {len(tcodes)}')

Building ATT&CK KB entries...
  3463 entries loaded

Entries per source:
  MITRE                 564
  OTRF                    4
  Sigma                2862
  Splunk                 33

Unique T-codes: 603


## 3. Embedding model comparison

Before committing to a ChromaDB index we compare several `sentence-transformers` models on ATT&CK retrieval recall.

For each model a temporary ChromaDB collection is created under `checkpoints/emb_eval/` and evaluated on a sample of labelled OTRF Atomic and Splunk chains.  The best-performing model is stored in `BEST_MODEL` and used to build the production ChromaDB in the next section.

In [6]:
import pickle, random, re
from collections import Counter
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions

import torch
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Embedding device: {_DEVICE}')

EVAL_CKPT  = Path('checkpoints')
EVAL_MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",   # current baseline
    "BAAI/bge-base-en-v1.5",                     # top retrieval benchmarks
    "sentence-transformers/all-mpnet-base-v2",   # strong general quality
    "intfloat/e5-base-v2",                       # passage retrieval focused
]
K_VALUES   = [5, 10, 15]
SEED       = 42

_SKIP = {"", "-1", "nan", "None", "0", "-", "N/A", "n/a"}
EID_NAMES_E = {
    1:"Process Create",3:"Network Connection",7:"Image Load",8:"CreateRemoteThread",
    10:"Process Access",11:"File Create",12:"Registry Create/Delete",
    13:"Registry Value Set",17:"Pipe Created",18:"Pipe Connected",
    22:"DNS Query",23:"File Delete",25:"Process Tampering",
}
SOURCE_BOUNDS_E = {
    "otrf_at": (511_105,   1_054_182),
    "splunk":  (1_382_201, 3_447_667),
}

# ── Helpers ──────────────────────────────────────────────────────────────────
def _filter_chains(chains, lo, hi):
    return [c for c in chains if len(c)>0 and int(c.min())>=lo and int(c.max())<hi]

def _chain_gt(chain, df):
    techs = [str(df.iloc[r].get("attck_technique","")).strip()
             for r in list(chain) if r < len(df)]
    techs = [t for t in techs if t and t not in _SKIP]
    return Counter(techs).most_common(1)[0][0] if techs else ""

def _build_query(rows, df):
    eid_c = Counter()
    images, targets, cmdlines = [], [], []
    for r in rows:
        if r >= len(df): continue
        ev = df.iloc[r]
        eid = ev.get("event_id","")
        if str(eid).isdigit(): eid_c[int(eid)] += 1
        for lst, fld in [(images,"image"),(targets,"target_object"),
                         (targets,"target_image"),(cmdlines,"command_line")]:
            v = str(ev.get(fld,"")).strip()
            if v and v not in _SKIP: lst.append(v)
    eid_s = ", ".join(f'{EID_NAMES_E.get(e,f"EID={e}")} x{n}'
                      for e,n in sorted(eid_c.items(),key=lambda x:-x[1]))
    def _uniq(lst,n=3):
        seen,out=[],[]
        for v in lst:
            k=v.split("\\")[-1].lower()
            if k not in seen: seen.append(k);out.append(v)
            if len(out)>=n: break
        return out
    parts=[f"Suspicious Windows Sysmon activity — event types: {eid_s}."]
    if images:  parts.append("Key processes: "+", ".join(_uniq(images)))
    if cmdlines: parts.append("Command lines: "+"; ".join(_uniq(cmdlines,2)))
    if targets:  parts.append("Targets: "+", ".join(_uniq(targets)))
    return " ".join(parts)

# ── Build test set ────────────────────────────────────────────────────────────
print("Loading chains and events_m...")
events_m_e = pd.read_parquet(EVAL_CKPT / "events_m.parquet")
with open(EVAL_CKPT / "seq" / "seq_chains_m.pkl","rb") as f:
    chains_m_e = pickle.load(f)

rng = random.Random(SEED)
test_cases = []
for src,(lo,hi) in SOURCE_BOUNDS_E.items():
    src_chains = _filter_chains(chains_m_e, lo, hi)
    labelled   = [(c,gt) for c in src_chains if (gt:=_chain_gt(c,events_m_e))]
    for chain,gt in labelled:
        rows = list(chain)
        if len(rows)>300: rows=sorted(rng.sample(rows,300))
        test_cases.append({"source":src,"gt":gt,"query":_build_query(rows,events_m_e)})

print(f"Test set: {len(test_cases)} chains")

# ── Evaluate each model ───────────────────────────────────────────────────────
from dataclasses import asdict
from data.attack_kb.builder import build_kb as _build_kb_e
entries_e = [asdict(e) for e in _build_kb_e(force=False)]

results_rows = []
for model_name in EVAL_MODELS:
    safe = re.sub(r"[^a-zA-Z0-9_-]","_",model_name)
    persist = str(EVAL_CKPT / "emb_eval" / safe)
    client  = chromadb.PersistentClient(path=persist)
    emb_fn  = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name, device=_DEVICE)
    coll    = client.get_or_create_collection("attackkb",embedding_function=emb_fn,
                                              metadata={"hnsw:space":"cosine"})
    if coll.count()==0:
        print(f"{model_name.split('/')[-1]}: ingesting {len(entries_e)} docs...",
              end=" ",flush=True)
        ids=[f"{e['source']}_{e['source_id']}_{i}" for i,e in enumerate(entries_e)]
        docs=[e["description"] for e in entries_e]
        metas=[{"technique_id":e["technique_id"],"technique_name":e["technique_name"]}
               for e in entries_e]
        for s in range(0,len(ids),500):
            coll.add(ids=ids[s:s+500],documents=docs[s:s+500],metadatas=metas[s:s+500])
        print("done.")
    else:
        print(f"{model_name.split('/')[-1]}: using cached collection ({coll.count()} docs)")

    recall = {}
    kmax = max(K_VALUES)
    for tc in test_cases:
        res = coll.query(query_texts=[tc["query"]],n_results=min(kmax,coll.count()))
        hit_ids = [m.get("technique_id","") for m in res["metadatas"][0]]
        for k in K_VALUES:
            recall.setdefault(k,[]).append(tc["gt"] in hit_ids[:k])

    row = {"model": model_name.split("/")[-1]}
    row.update({f"recall@{k}": f'{sum(v)/len(v):.1%}' for k,v in recall.items()})
    results_rows.append(row)

# ── Results table ─────────────────────────────────────────────────────────────
print()
df_emb = pd.DataFrame(results_rows).set_index("model")
print(df_emb.to_string())

best_row = max(results_rows,
               key=lambda r: float(r[f"recall@{max(K_VALUES)}"].rstrip("%"))/100)
best_full = next(m for m in EVAL_MODELS if m.split("/")[-1]==best_row["model"])
print(f" Best: {best_full}  ({best_row[f'recall@{max(K_VALUES)}']} @ k={max(K_VALUES)})")
# Expose best model for the next cell
BEST_MODEL = best_full
print(f"\nBEST_MODEL = {BEST_MODEL!r}")

Embedding device: cuda
Loading chains and events_m...
Test set: 17750 chains


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


all-MiniLM-L6-v2: using cached collection (601 docs)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


all-mpnet-base-v2: using cached collection (601 docs)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


e5-base-v2: using cached collection (601 docs)

                  recall@5 recall@10 recall@15
model                                         
all-MiniLM-L6-v2      2.2%      7.8%     11.9%
bge-base-en-v1.5      4.4%     11.2%     13.9%
all-mpnet-base-v2     7.2%     12.2%     16.3%
e5-base-v2            5.3%      9.3%     12.7%
 Best: sentence-transformers/all-mpnet-base-v2  (16.3% @ k=15)

BEST_MODEL = 'sentence-transformers/all-mpnet-base-v2'


## 4b. Dense vs Hybrid recall comparison

Using the `emb_eval/` temp collection already built for `BEST_MODEL` in cell 10 — no production ChromaDB needed yet.  
BM25 is built in-memory from the same KB entries.  
Recall@k = fraction of chains where the correct ATT&CK technique appears in the top-k candidates.  This is the ceiling on LLM attribution accuracy.

In [7]:
# Dense vs Hybrid comparison using the emb_eval/ temp collection from cell 10.
# No production ChromaDB needed here — we reuse what the embedding comparison built.
import pickle, random, re
from collections import Counter
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

import torch
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Embedding device: {_DEVICE}')

SEED     = 42
K_VALUES = [5, 10, 15]
SOURCE_BOUNDS_E = {
    "otrf_at": (511_105,   1_054_182),
    "splunk":  (1_382_201, 3_447_667),
}
_SKIP_E = {"", "-1", "nan", "None", "0", "-", "N/A", "n/a"}
EID_NAMES_E = {
    1:"Process Create", 3:"Network Connection", 7:"Image Load",
    8:"CreateRemoteThread", 10:"Process Access", 11:"File Create",
    12:"Registry Create/Delete", 13:"Registry Value Set",
    17:"Pipe Created", 18:"Pipe Connected", 22:"DNS Query",
    23:"File Delete", 25:"Process Tampering",
}


def _filter_chains_e(chains, lo, hi):
    return [c for c in chains if len(c) > 0 and int(c.min()) >= lo and int(c.max()) < hi]


def _chain_gt_e(chain, df):
    techs = [str(df.iloc[r].get("attck_technique", "")).strip()
             for r in list(chain) if r < len(df)]
    techs = [t for t in techs if t and t not in _SKIP_E]
    return Counter(techs).most_common(1)[0][0] if techs else ""


def _build_query_e(rows, df):
    eid_c = Counter()
    images, targets, cmdlines = [], [], []
    for r in rows:
        if r >= len(df): continue
        ev = df.iloc[r]
        eid = ev.get("event_id", "")
        if str(eid).isdigit(): eid_c[int(eid)] += 1
        for lst, fld in [(images, "image"), (targets, "target_object"),
                         (targets, "target_image"), (cmdlines, "command_line")]:
            v = str(ev.get(fld, "")).strip()
            if v and v not in _SKIP_E: lst.append(v)
    eid_s = ", ".join(f'{EID_NAMES_E.get(e, f"EID={e}")} x{n}'
                      for e, n in sorted(eid_c.items(), key=lambda x: -x[1]))
    def _uniq(lst, n=3):
        seen, out = [], []
        for v in lst:
            k = v.split("\\")[-1].lower()
            if k not in seen: seen.append(k); out.append(v)
            if len(out) >= n: break
        return out
    parts = [f"Suspicious Windows Sysmon activity - event types: {eid_s}."]
    if images:   parts.append("Key processes: " + ", ".join(_uniq(images)))
    if cmdlines: parts.append("Command lines: " + "; ".join(_uniq(cmdlines, 2)))
    if targets:  parts.append("Targets: " + ", ".join(_uniq(targets)))
    return " ".join(parts)


def _tokenize_e(text):
    return re.findall(r'[a-z0-9]+(?:[._][a-z0-9]+)*', text.lower())


# ── Load test set ─────────────────────────────────────────────────────────────
print("Loading chains and events_m...")
CKPT_DIR_E = Path("checkpoints")
events_m_e = pd.read_parquet(CKPT_DIR_E / "events_m.parquet")
with open(CKPT_DIR_E / "seq" / "seq_chains_m.pkl", "rb") as f:
    chains_m_e = pickle.load(f)

rng_e = random.Random(SEED)
test_cases_e = []
for src, (lo, hi) in SOURCE_BOUNDS_E.items():
    src_chains = _filter_chains_e(chains_m_e, lo, hi)
    labelled   = [(c, gt) for c in src_chains if (gt := _chain_gt_e(c, events_m_e))]
    for chain, gt in labelled:
        rows = list(chain)
        if len(rows) > 300: rows = sorted(rng_e.sample(rows, 300))
        test_cases_e.append({"source": src, "gt": gt,
                              "query": _build_query_e(rows, events_m_e)})

print(f"Test set: {len(test_cases_e)} chains")

# ── Open the emb_eval/ ChromaDB for BEST_MODEL (built in cell 10) ─────────────
safe_name = re.sub(r"[^a-zA-Z0-9_-]", "_", BEST_MODEL)
persist   = str(CKPT_DIR_E / "emb_eval" / safe_name)
emb_fn    = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=BEST_MODEL, device=_DEVICE)
coll      = chromadb.PersistentClient(path=persist).get_collection(
                "attackkb", embedding_function=emb_fn)
print(f"Temp collection : {persist}  ({coll.count()} docs)")

# ── Build in-memory BM25 from the same KB entries ─────────────────────────────
from dataclasses import asdict
from data.attack_kb.builder import build_kb as _bkb

entries_e  = _bkb(force=False)
bm25_docs  = [e.description for e in entries_e]
bm25_meta  = [{"technique_id": e.technique_id, "technique_name": e.technique_name}
              for e in entries_e]
bm25_model = BM25Okapi([_tokenize_e(d) for d in bm25_docs])
print(f"BM25 index      : {len(bm25_docs)} docs (in-memory)")

# ── Retrieval helpers ─────────────────────────────────────────────────────────
RRF_K = 60


def _retrieve_dense(query, k):
    kmax = min(k * 4, coll.count())
    res  = coll.query(query_texts=[query], n_results=kmax)
    return [m.get("technique_id", "") for m in res["metadatas"][0]]


def _retrieve_bm25(query, k):
    scores  = bm25_model.get_scores(_tokenize_e(query))
    top_idx = np.argsort(scores)[::-1][:k * 4]
    return [bm25_meta[i]["technique_id"] for i in top_idx if scores[i] > 0]


def _retrieve_hybrid(query, k):
    dense_ids = _retrieve_dense(query, k)
    bm25_ids  = _retrieve_bm25(query, k)
    rrf: dict = {}
    for rank, tid in enumerate(dense_ids):
        rrf[tid] = rrf.get(tid, 0.0) + 1.0 / (RRF_K + rank + 1)
    for rank, tid in enumerate(bm25_ids):
        rrf[tid] = rrf.get(tid, 0.0) + 1.0 / (RRF_K + rank + 1)
    return [tid for tid, _ in sorted(rrf.items(), key=lambda x: -x[1])]


# ── Evaluate ──────────────────────────────────────────────────────────────────
kmax = max(K_VALUES)
recall_dense  = {k: [] for k in K_VALUES}
recall_hybrid = {k: [] for k in K_VALUES}

for i, tc in enumerate(test_cases_e):
    if i % 200 == 0:
        print(f"  {i}/{len(test_cases_e)}...", end="\r", flush=True)
    d_ids = _retrieve_dense(tc["query"],  kmax)
    h_ids = _retrieve_hybrid(tc["query"], kmax)
    for k in K_VALUES:
        recall_dense[k].append(tc["gt"] in d_ids[:k])
        recall_hybrid[k].append(tc["gt"] in h_ids[:k])

n = len(test_cases_e)
model_short = BEST_MODEL.split("/")[-1]
hdr = f"{'Method':<34}" + "  ".join(f"recall@{k}" for k in K_VALUES)
row_d = f"{'Dense ('+model_short+')':<34}" + "  ".join(
    f"{sum(recall_dense[k])/n:.1%}    " for k in K_VALUES)
row_h = f"{'Hybrid (dense + BM25 RRF)':<34}" + "  ".join(
    f"{sum(recall_hybrid[k])/n:.1%}    " for k in K_VALUES)
print(f"\n{hdr}\n{row_d}\n{row_h}")

Embedding device: cuda
Loading chains and events_m...
Test set: 17750 chains
Temp collection : checkpoints\emb_eval\sentence-transformers_all-mpnet-base-v2  (601 docs)
BM25 index      : 3463 docs (in-memory)
  17600/17750...
Method                            recall@5  recall@10  recall@15
Dense (all-mpnet-base-v2)         7.2%      12.3%      16.4%    
Hybrid (dense + BM25 RRF)         11.9%      18.2%      25.4%    


## 5. Build ChromaDB with best model

Using `BEST_MODEL` selected above, update `config.py` and re-ingest the full ATT&CK KB into the production ChromaDB collection.

In [8]:
import importlib, re as _re
from pathlib import Path as _Path

# --- 1. Update config.py on disk ---------------------------------------------
cfg_path = _Path('..') / 'config.py'
cfg_text  = cfg_path.read_text(encoding='utf-8')
cfg_updated = _re.sub(
    r'(EMBEDDING_MODEL\s*=\s*)["\'\'][^\"\'\']*["\'\']',
    f'\\1"{BEST_MODEL}"',
    cfg_text,
)
if cfg_updated != cfg_text:
    cfg_path.write_text(cfg_updated, encoding='utf-8')
    print(f'  config.py updated : EMBEDDING_MODEL = "{BEST_MODEL}"')
else:
    print(f'  config.py already : EMBEDDING_MODEL = "{BEST_MODEL}"')

# --- 2. Reload config + vector_store so they pick up the new model -----------
import config as _cfg
importlib.reload(_cfg)

import data.attack_kb.vector_store as _vs
importlib.reload(_vs)

from data.attack_kb.vector_store import ingest as _ingest, get_collection as _gc

# --- 3. Re-ingest ChromaDB with the best model --------------------------------
print(f'\nIngesting into ChromaDB...')
print(f'  Embedding model : {BEST_MODEL}')
print(f'  Persist dir     : {_cfg.CHROMA_PERSIST_DIR}')

_ingest(entries, force=True)   # force=True to rebuild with new embedding model

col = _gc()
print(f'  Collection size : {col.count()} documents')
print('Done.')

  config.py already : EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

Ingesting into ChromaDB...
  Embedding model : sentence-transformers/all-mpnet-base-v2
  Persist dir     : H:\Challenge-3-4\cyber-anomaly-detection\data\attack_kb\chroma_db
  Collection size : 3463 documents
Done.


## 6. Smoke-test retrieval

In [9]:
# Re-import retrieve after module reload in previous cell
from data.attack_kb.vector_store import retrieve

QUERIES = [
    # (description, expected_technique_prefix)
    ('mimikatz lsass credential dumping process memory',        'T1003'),
    ('powershell encoded command execution bypass',             'T1059'),
    ('scheduled task persistence registry run key',            'T1053'),
    ('lateral movement psexec smb remote service',             'T1021'),
    ('dns query exfiltration tunneling covert channel',        'T1071'),
]

for query, expected in QUERIES:
    hits = retrieve(query, k=RAG_TOP_K)
    top = hits[0] if hits else {}
    tid   = top.get('technique_id', 'N/A')
    tname = top.get('technique_name', '')
    dist  = top.get('distance', 1.0)
    match = '✓' if tid.startswith(expected) else '✗'
    print(f'{match} [{tid}] {tname[:45]:<45}  dist={dist:.3f}')
    print(f'  query: {query}')
print()
print('=== Hybrid retrieval smoke-test (dense + BM25 RRF) ===')
from data.attack_kb.vector_store import retrieve_hybrid
for query, expected in QUERIES:
    hits = retrieve_hybrid(query, k=RAG_TOP_K)
    top  = hits[0] if hits else {}
    tid  = top.get('technique_id', 'N/A')
    tname = top.get('technique_name', '')
    dist  = top.get('distance', 1.0)
    match = '✓' if tid.startswith(expected) else '✗'
    print(f'{match} [{tid}] {tname[:45]:<45}  dist={dist:.3f}')
    print(f'  query: {query}')

✓ [T1003.001] LSASS Memory                                   dist=0.284
  query: mimikatz lsass credential dumping process memory
✗ [T1027] Potential Encoded PowerShell Patterns In Comm  dist=0.352
  query: powershell encoded command execution bypass
✓ [T1053.005] Scheduled Task                                 dist=0.396
  query: scheduled task persistence registry run key
✗ [T1069.001] Suspicious Get Information for SMB Share - Po  dist=0.498
  query: lateral movement psexec smb remote service
✓ [T1071.004] DNS                                            dist=0.313
  query: dns query exfiltration tunneling covert channel

=== Hybrid retrieval smoke-test (dense + BM25 RRF) ===
✓ [T1003.001] Credential Dumping Attempt Via WerFault        dist=0.428
  query: mimikatz lsass credential dumping process memory
✓ [T1059.001] PowerShell Base64 Encoded Invoke Keyword       dist=0.396
  query: powershell encoded command execution bypass
✓ [T1053.005] Scheduled Task                                

## 7. Verify index lookup (end-to-end)

In [ ]:
# Simulate what notebook 10 will do: given flagged row indices,
# retrieve raw event text and run retrieval.
#
# We use an OTRF Atomic chain here (known T-code, populated fields) rather
# than an LMD chain (no T-code; LMD stores empty fields as the literal "0").

import pickle
from pathlib import Path

SEQ_DIR  = Path('checkpoints') / 'seq'
CKPT_DIR = Path('checkpoints')

SOURCE_BOUNDS = {
    'lmd':     (0,           511_105),
    'otrf_at': (511_105,   1_054_182),
    'otrf_cp': (1_054_182, 1_382_201),
    'splunk':  (1_382_201, 3_447_667),
}

with open(SEQ_DIR / 'seq_chains_m.pkl', 'rb') as f:
    chains_m = pickle.load(f)

def filter_chains_by_source(chains, lo, hi):
    return [c for c in chains if len(c) > 0
            and int(c.min()) >= lo and int(c.max()) < hi]

# Pick longest OTRF Atomic chain — has real T-code ground truth + populated fields
lo, hi = SOURCE_BOUNDS['otrf_at']
otrf_chains = filter_chains_by_source(chains_m, lo, hi)
longest = max(otrf_chains, key=len)

W = 45
window_rows = list(longest[:W])

print(f'Source          : OTRF Atomic')
print(f'Window rows     : {window_rows[0]}..{window_rows[-1]}')
print(f'Technique (GT)  : {events_m.iloc[window_rows[0]]["attck_technique"]}')
print()


def format_events_for_slm(rows: list[int], df) -> str:
    """Format raw Sysmon events as structured text for the SLM."""
    # Values excluded as uninformative (LMD stores empty as "0"; -1 is numeric default)
    _SKIP = {'', '-1', 'nan', 'None', '0', '-'}
    lines = []
    for i, r in enumerate(rows):
        ev = df.iloc[r]
        parts = [f'Event {i+1}: EID={ev.get("event_id", "")}']
        for field in ['image', 'command_line', 'parent_image', 'parent_cmdline',
                      'target_object', 'target_image',
                      'dest_ip', 'dest_hostname', 'target_filename', 'query_name']:
            val = str(ev.get(field, '')).strip()
            if val and val not in _SKIP:
                parts.append(f'  {field}: {val[:120]}')
        lines.append('\n'.join(parts))
    return '\n---\n'.join(lines)


event_text = format_events_for_slm(window_rows, events_m)
print('=== Formatted event window (first 1500 chars) ===')
print(event_text[:1500])
print()

# Retrieve top-3 matching techniques
from data.attack_kb.vector_store import retrieve
hits = retrieve(event_text[:500], k=3)
print('=== Top-3 retrieved techniques ===')
for h in hits:
    print(f'  [{h["technique_id"]}] {h["technique_name"][:50]:<50}  dist={h["distance"]:.3f}')

Source          : OTRF Atomic
Window rows     : 603093..603101
Technique (GT)  : T1562.002

=== Formatted event window (first 1500 chars) ===
Event 1: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\winlogon.exe
---
Event 2: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 3: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 4: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 5: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 6: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 7: EID=10
  image: C:\Windows\System32\VBoxService.exe
  target_image: C:\Windows\system32\svchost.exe
---
Event 8: EID=10
  image: C:\Windows\System32\VBoxService.exe
  tar

: 